# Kafka issue types: exploration and BiLSTM

In [ ]:
import re
import pandas as pd
from jira import JIRA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import gensim.downloader as gensim_api
from tqdm.auto import tqdm


jira = JIRA('https://issues.apache.org/jira')

jql = 'project=KAFKA'

all_issues = []
start = 0
batch_size = 100

while True:
    issues = jira.search_issues(
        jql,
        startAt=start,
        maxResults=batch_size,
        fields=['summary', 'issuetype', 'status', 'created', 'reporter', 'description'])
    if not issues:
        break
    all_issues.extend(issues)
    print(f"Found {len(all_issues)} issues")
    start += batch_size
print(f"Found {len(all_issues)} issues total")


def extract_issue(issue):
    return {
        'bug': issue.key,
        'summary': issue.fields.summary or '',
        'description': issue.fields.description or '',
        'type': issue.fields.issuetype.name,
        'status': issue.fields.status.name,
        'created': issue.fields.created,
        'reporter': issue.fields.reporter.displayName if issue.fields.reporter else 'Unknown'
    }


data = [extract_issue(issue) for issue in all_issues]
df = pd.DataFrame(data)
df.to_csv('Kafka_issues.csv', index=False)



print(f"\nTotal issues: {len(df)}")

print("\nIssue type distribution:")
print(df['type'].value_counts())
print("\nIssue type distribution (%):")
print((df['type'].value_counts(normalize=True) * 100).round(2))

print("\nStatus distribution:")
print(df['status'].value_counts())

print("\nMissing/empty descriptions:")
missing_desc = df['description'].isna() | (df['description'].fillna('').str.strip() == '')
print(f"{missing_desc.sum()} of {len(df)} issues ({missing_desc.mean()*100:.1f}%) have no description")

print("\nTop 10 reporters by issue count:")
print(df['reporter'].value_counts().head(10))

summary_lengths = df['summary'].fillna('').str.split().str.len()
desc_lengths = df['description'].fillna('').str.split().str.len()
print(f"\nSummary length (words): mean={summary_lengths.mean():.1f}, "
      f"median={summary_lengths.median():.0f}, max={summary_lengths.max()}")
print(f"Description length (words): mean={desc_lengths.mean():.1f}, "
      f"median={desc_lengths.median():.0f}, max={desc_lengths.max()}")

df['created_date'] = pd.to_datetime(df['created'])
print(f"\nDate range: {df['created_date'].min().date()} to {df['created_date'].max().date()}")
print("\nIssues created per year:")
print(df['created_date'].dt.year.value_counts().sort_index())


df['text'] = (df['summary'].fillna('') + ' ' + df['description'].fillna('')).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)

MIN_CLASS_COUNT = 50
type_counts = df['type'].value_counts()
rare_types = type_counts[type_counts < MIN_CLASS_COUNT].index
if len(rare_types) > 0:
  print(f"Collapsing rare issue types into 'Other' (<50 ex.): "
          f"{list(rare_types)}")
  df['type'] = df['type'].apply(lambda t: 'Other' if t in rare_types else t)

categories = sorted(df['type'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(categories)}
df['label'] = df['type'].map(label_to_idx)

print(f"\nModeling categories: {categories}")
print(df['type'].value_counts())

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df['label']
)


class IssueDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


train_dataset = IssueDataset(train_texts, train_labels)
test_dataset = IssueDataset(test_texts, test_labels)


def tokenizer(text):
    return re.findall(r'\b[a-zA-Z]+\b', text.lower())


def build_vocab(dataset, min_freq=2):
    word_counts = {}
    for text, _ in dataset:
        for word in tokenizer(text):
            word_counts[word] = word_counts.get(word, 0) + 1
    vocab = {'<pad>': 0, '<unk>': 1}
    idx = 2
    for word, count in word_counts.items():
        if count >= min_freq:
            vocab[word] = idx
            idx += 1
    return vocab


vocab = build_vocab(train_dataset)
vocab_size = len(vocab)
print(f"\nVocabulary size: {vocab_size}")

embedding_dim = 100
glove_vectors = gensim_api.load("glove-wiki-gigaword-100")

embedding_matrix = torch.zeros(vocab_size, embedding_dim)
hits = 0
for word, idx in vocab.items():
    if word in glove_vectors.key_to_index:
        embedding_matrix[idx] = torch.tensor(glove_vectors[word])
        hits += 1
    else:
        embedding_matrix[idx] = torch.randn(embedding_dim) * 0.1

del glove_vectors


def collate_batch(batch):
    labels, texts, lengths = [], [], []
    for text, label in batch:
        labels.append(label)
        tokens = [vocab.get(word, vocab['<unk>']) for word in tokenizer(text)]
        if len(tokens) == 0:
            tokens = [vocab['<unk>']]
        texts.append(torch.tensor(tokens))
        lengths.append(len(tokens))
    texts = nn.utils.rnn.pad_sequence(texts, batch_first=True, padding_value=vocab['<pad>'])
    return texts, torch.tensor(labels), torch.tensor(lengths)


batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding.weight.data.copy_(embedding_matrix)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, (hidden, cell) = self.lstm(packed)
        hidden_forward = hidden[0]
        hidden_backward = hidden[1]
        combined = torch.cat((hidden_forward, hidden_backward), dim=1)
        return self.fc(combined)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = LSTM(vocab_size, embedding_dim, 128, len(categories)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
print("Starting training...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for texts, labels, lengths in progress:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts, lengths)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for texts, labels, lengths in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        output = model(texts, lengths)
        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels)
macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=categories))

Found 19446 issues total

Total issues: 19446

Issue type distribution:
type
Bug            8874
Improvement    5002
Sub-task       3314
Task            869
Test            648
New Feature     606
Wish            133
Name: count, dtype: int64

Issue type distribution (%):
type
Bug            45.63
Improvement    25.72
Sub-task       17.04
Task            4.47
Test            3.33
New Feature     3.12
Wish            0.68
Name: proportion, dtype: float64

Status distribution:
status
Resolved           13490
Open                4541
Closed               772
In Progress          288
Patch Available      283
Reopened              72
Name: count, dtype: int64

Missing/empty descriptions:
1432 of 19446 issues (7.4%) have no description

Top 10 reporters by issue count:
reporter
Chia-Ping Tsai            968
Jason Gustafson           596
Matthias J. Sax           551
Jun Rao                   474
Guozhang Wang             459
Ismael Juma               400
Colin McCabe              337
Mickael

Epoch 1/10, Loss: 1.1887


Epoch 2/10, Loss: 0.9919


Epoch 3/10, Loss: 0.8796


Epoch 4/10, Loss: 0.7477


Epoch 5/10, Loss: 0.6127


Epoch 6/10, Loss: 0.4756


Epoch 7/10, Loss: 0.3367


Epoch 8/10, Loss: 0.2333


Epoch 9/10, Loss: 0.1423


Epoch 10/10, Loss: 0.0843

Accuracy: 0.5995
Macro F1: 0.3812
Weighted F1: 0.5977

Classification Report:
              precision    recall  f1-score   support

         Bug       0.77      0.72      0.74      1775
 Improvement       0.46      0.59      0.52      1001
 New Feature       0.27      0.18      0.22       121
    Sub-task       0.61      0.56      0.58       663
        Task       0.21      0.13      0.16       174
        Test       0.38      0.40      0.39       130
        Wish       0.17      0.04      0.06        26

    accuracy                           0.60      3890
   macro avg       0.41      0.37      0.38      3890
weighted avg       0.60      0.60      0.60      3890

